In [1]:
import warnings
warnings.filterwarnings("ignore")
from yfinance import download
from pandas import read_csv, DataFrame, concat
import plotly.express as px
from numpy import arange, polyfit

In [2]:
path = r"Q:\financial_data_pipelines\data\pipelines\b3_indices_segmentos_setoriais\processed\transform_1\IBEP.csv"
df = read_csv(path)
df.tail(3)

,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
73,VIVA3,VIVARA S.A.,ON NM,123.160.591,"0,208"
74,WEGE3,WEG,ON EDJ NM,1.485.954.732,"3,521"
75,YDUQ3,YDUQS PART,ON NM,260.753.144,"0,156"


In [3]:
tickers = df['Código']

In [4]:
df_series = DataFrame()

for i, ticker in enumerate(list(tickers)):
    
    try:
        series = download(
            f'{ticker}.SA',
            # start="2010-01-01",
            # end="2020-01-27",
            interval='1d',
            period='10y',
            progress=False,
            auto_adjust=False
            ).droplevel(1, axis=1)[['Close']].rename(columns={'Close': ticker})
    except:

        continue

    
    df_series = concat([series, df_series], axis=1)


1 Failed download:
['ALOS3.SA']: DNSError('Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')

1 Failed download:
['ABEV3.SA']: DNSError('Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')

1 Failed download:
['ASAI3.SA']: DNSError('Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')

1 Failed download:
['AURE3.SA']: DNSError('Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')

1 Failed download:
['AXIA3.SA']: DNSError('Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')

1 Fa

In [5]:
df_series.tail(3)

Price,YDUQ3,WEGE3,VIVA3,VBBR3,VAMO3,VALE3,USIM5,UGPA3,TOTS3,TIMS3,...,BBDC4,BBDC3,B3SA3,AZZA3,AXIA6,AXIA3,AURE3,ASAI3,ABEV3,ALOS3
Date,,,,,,,,,,,,,,,,,,,,,


In [6]:
# normalização dos preços

df_norm = df_series.copy()

for col in df_norm.columns:
    serie = df_norm[col]

    base = serie.dropna()
    if len(base) < 2:
        continue

    df_norm[col] = serie / base.iloc[0]

In [7]:
df_norm.tail(3)

Price,YDUQ3,WEGE3,VIVA3,VBBR3,VAMO3,VALE3,USIM5,UGPA3,TOTS3,TIMS3,...,BBDC4,BBDC3,B3SA3,AZZA3,AXIA6,AXIA3,AURE3,ASAI3,ABEV3,ALOS3
Date,,,,,,,,,,,,,,,,,,,,,


In [8]:
fig = px.line(
    df_norm,
    x=df_norm.index,
    y=df_norm.columns
)

fig.update_layout(
    template="plotly_white",
    width=1200,
    height=700
)

fig.show()


ValueError: Cannot accept list of column references or list of columns for both `x` and `y`.

In [ ]:
# calculos

def regressao_linear(x, y):
    x = arange(x)
    y = y
    coef = polyfit(x, y, 1)
    return coef[0] * x + coef[1]

In [ ]:
# distancia da regressao linear
def calculo_1(serie, col):
    serie['regressao_linear'] = regressao_linear(x=len(serie), y=serie[col].values)
    serie['dif_regressao_linear'] = serie[col] - serie['regressao_linear']
    value = serie['dif_regressao_linear'].iloc[-1]
    return value

# distancia da media de 200
def calculo_2(serie, col):
    serie['media_a_200'] = serie[col].rolling(200).mean()
    serie['dif_media_a_200'] = serie[col] - serie['media_a_200']
    value = serie['dif_media_a_200'].iloc[-1]
    return value


In [ ]:
df_calc = df_series.copy()

dic = {}

for col in df_calc.columns:
    serie = df_calc[[col]]
    serie = serie.dropna()

    try:
        calc_1 = calculo_1(serie, col)
        calc_2 = calculo_2(serie, col)
    except:
        print(serie)
        
    dic[col] = [
        calc_1, 
        calc_2
        ]

In [ ]:
df_calc = DataFrame(dic).T

df_calc.tail(3)

,0,1
ASAI3,-0.936375,-2.205800
ABEV3,1.651350,0.453250
ALOS3,10.278891,4.809651


In [ ]:

df_calc = df_calc.reset_index()
df_calc.columns = [
    "ativo", 
    "distancia_da_regressao_linear",
    "distancia_da_media_a_200"
    ]


df_calc

,ativo,distancia_da_regressao_linear,distancia_da_media_a_200
0,YDUQ3,-3.405881,-1.248250
1,WEGE3,-1.860630,6.527998
2,VIVA3,6.603202,6.043300
3,VBBR3,6.318498,4.603225
4,VAMO3,-1.079955,-0.800650
...,...,...,...
71,AXIA3,1.125244,2.424299
72,AURE3,2.805728,1.911750
73,ASAI3,-0.936375,-2.205800
74,ABEV3,1.651350,0.453250


In [ ]:

# Ordena para leitura visual
df_plot = df_calc.sort_values("distancia_da_regressao_linear")

media = df_plot["distancia_da_regressao_linear"].mean()

fig = px.bar(
    df_plot,
    x="ativo",
    y="distancia_da_regressao_linear",
    title="Distância em relação à regressão linear por ativo",
)

# Linha da média
fig.add_hline(
    y=media,
    line_dash="dash",
    line_width=2,
    line_color="black",
    annotation_text=f"Média: {media:.2f}",
    annotation_position="top right",
    annotation_font=dict(
        size=14,                 
        color="black",         
        family="Arial Black"     
    )
)

# Layout 
fig.update_layout(
    template="plotly_white",
    width=1200,
    height=650,
    xaxis_tickangle=-90,
    xaxis_title="Ativo",
    yaxis_title="Distância da regressão linear",
    title={
        "x": 0.01,
        "xanchor": "left",
        "font": {"size": 18}
    },
    font={"size": 12},
    bargap=0.15,
    hovermode="x unified",
)

# Grid sutil
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")

fig.update_traces(
    marker_color="#73C0FF"  # azul profissional
)

fig.show()


In [ ]:
qtd_ativo_acima_rl = len(df_calc[df_calc['distancia_da_regressao_linear'] > 0])
qtd_ativo_abaixo_rl = len(df_calc[df_calc['distancia_da_regressao_linear'] < 0])

print(f"Quantidade de ativos com preços acima da regressão linear: ", qtd_ativo_acima_rl)
print(f"Quantidade de ativos com preços abaixo da regressão linear: ", qtd_ativo_abaixo_rl)

Quantidade de ativos com preços acima da regressão linear:  39
Quantidade de ativos com preços abaixo da regressão linear:  37


In [ ]:

# Ordena para leitura visual
df_plot = df_calc.sort_values("distancia_da_media_a_200")

media = df_plot["distancia_da_media_a_200"].mean()

fig = px.bar(
    df_plot,
    x="ativo",
    y="distancia_da_media_a_200",
    title="Distância em relação à média de 200 periodos por ativo",
)

# Linha da média
fig.add_hline(
    y=media,
    line_dash="dash",
    line_width=2,
    line_color="black",
    annotation_text=f"Média: {media:.2f}",
    annotation_position="top right",
    annotation_font=dict(
        size=14,                 
        color="black",         
        family="Arial Black"     
    )
)

# Layout 
fig.update_layout(
    template="plotly_white",
    width=1400,
    height=650,
    xaxis_tickangle=-90,
    xaxis_title="Ativo",
    yaxis_title="Distância em relação à média de 200 periodos",
    title={
        "x": 0.01,
        "xanchor": "left",
        "font": {"size": 18}
    },
    font={"size": 12},
    bargap=0.15,
    hovermode="x unified",
)

# Grid sutil
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")

fig.update_traces(
    marker_color="#C773FF"  # azul profissional
)

fig.show()
